# **Genome Annotation – Prokka**

## **Tool Information**

- **Tool:** Prokka (v1.14.6)  
- **Input:** Assembled genome contigs (FASTA)  
- **Organism:** *Acinetobacter baumannii*  
- **Analysis type:** Structural and functional genome annotation  

Prokka is a rapid prokaryotic genome annotation pipeline used to identify genomic features such as coding sequences (CDS), rRNA, and tRNA genes. It integrates multiple tools and curated databases to assign functional annotations to predicted genes.

The pipeline uses Prodigal for gene prediction and performs similarity searches against trusted protein databases to assign gene names and functions. The output is standardized, making it suitable for downstream comparative genomic analyses.

This notebook documents the annotation of *Acinetobacter baumannii* genome assemblies to generate high-quality annotation files for further analyses such as pangenome construction and AMR gene context investigation.

## **Create a Dedicated Environment**

We create a dedicated conda environment to isolate Prokka and its dependencies from other tools. This ensures a stable and reproducible setup for genome annotation analyses.

In [ ]:
%%bash

conda create -n prokka_aba python=3.10 -y

## **Install Prokka using mamba**

We install Prokka using mamba for efficient dependency resolution and faster installation. The tool is installed from Bioconda along with required packages from conda-forge.

In [ ]:
%%bash

# Initialize conda
source /home/anaconda/miniconda3/etc/profile.d/conda.sh

# Activate environment
conda activate prokka_aba

# Install mamba
conda install -c conda-forge mamba -y

# Install Prokka
mamba install -c bioconda -c conda-forge prokka --channel-priority flexible -y

## **Verify Installation**

We verify that Prokka is installed correctly by checking the tool version. This confirms that the installation was successful and the tool is ready for genome annotation.

In [ ]:
%%bash

source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate prokka_aba

prokka --version

prokka 1.14.6


## **Input Requirements**

Prokka requires genome assemblies in FASTA format.

Each input file should contain:

1. Assembled contigs or complete genome sequence  
2. Properly formatted FASTA headers  
3. One genome per file  

Accepted formats include:

- `.fasta`  
- `.fa`  
- `.fna`  

Ensure that genome assemblies are:

- Free from contamination  
- Properly assembled and, if possible, polished  
- Consistently named for compatibility with downstream tools

## **Folder Structure Used**

Assemblies (input FASTA):
```/data/internship_data/nidhi/aba/output/nextflow_output/assemblies/```

Prokka output (per-sample folders):
```/data/internship_data/nidhi/aba/output/prokka_output/<SAMPLE>/```

Master log file:
```/data/internship_data/nidhi/aba/output/prokka_output/master_prokka.log```

Per-sample log files:
```<SAMPLE>/<SAMPLE>.log```

## **Execution Strategy**

Genome annotation is performed using a batch-processing approach, where each genome is annotated independently.

A consistent annotation scheme is maintained by specifying:

- Genus and species (*Acinetobacter baumannii*)  
- Use of genus-specific database (`--usegenus`)  
- Standardized output prefixes  

To annotate 650 *Acinetobacter baumannii* genomes efficiently, Prokka was executed using GNU Parallel with 12 concurrent jobs (each using 4 CPU cores). Parallel processing is used to improve computational efficiency by assigning multiple CPU threads.

Each genome generates a separate output directory containing annotation results.

In [ ]:
%%bash

source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate prokka

ls /data/internship_data/nidhi/aba/new_output/nextflow_output/assemblies/* | parallel -j 12 '
    fa={}
    sample=$(basename "$fa")
    sample="${sample%%.*}"
    outdir="/data/internship_data/nidhi/aba/new_output/prokka_output/$sample"

    mkdir -p "$outdir"

    prokka --force \
        --outdir "$outdir" \
        --prefix "$sample" \
        --cpus 4 \
        "$fa" > "$outdir/$sample.log" 2>&1

    ret=$?
    if [ $ret -ne 0 ]; then
        echo "PROKKA_FAILED $sample" >> /data/internship_data/nidhi/aba/new_output/prokka_output/master_prokka.log
    fi

    echo "=== COMPLETED $sample ===" >> /data/internship_data/nidhi/aba/new_output/prokka_output/master_prokka.log
'

## **Annotation Workflow**

### i) Gene Prediction and Functional Annotation

For each genome, Prokka performs:

- Prediction of coding sequences (CDS) using Prodigal  
- Identification of rRNA and tRNA genes  
- Functional annotation using curated protein databases  
- Assignment of standardized locus tags and gene names  

This ensures uniform annotation across all genomes, which is critical for comparative analysis.


### ii) Command Execution

#### Single Genome Annotation

```bash
prokka \
  --outdir output_genome1 \
  --prefix genome1 \
  --genus Acinetobacter \
  --species baumannii \
  --strain strain1 \
  --usegenus \
  --cpus 8 \
  input/genome1.fasta

## **Checking Prokka Success and Failure**

Each Prokka output folder should contain `<sample>.gff`among other files
The below code checks which samples succeeded and which failed.

In [11]:
%%bash

 PROKKA_DIR=/data/internship_data/nidhi/aba/new_output/prokka_output
 FAILED_LIST=$PROKKA_DIR/failed_samples.txt

 SUCCESS=0
 FAILED=0

 > $FAILED_LIST

 for d in "$PROKKA_DIR"/*; do
     [ -d "$d" ] || continue
     sample=$(basename "$d")
     gff="$d/$sample.gff"

     if [ -s "$gff" ]; then
         SUCCESS=$((SUCCESS+1))
     else
         echo "[FAILED] $sample"
         echo "$sample" >> $FAILED_LIST
         FAILED=$((FAILED+1))
     fi
 done

 echo
 echo "Total successful samples: $SUCCESS"
 echo "Total failed samples: $FAILED"
 echo "Failed list saved to: $FAILED_LIST"


Total successful samples: 71
Total failed samples: 0
Failed list saved to: /data/internship_data/nidhi/aba/new_output/prokka_output/failed_samples.txt


## **Expected Output Files**
Prokka generates annotated genome files in multiple formats for downstream genomic analyses. These outputs can be used for comparative genomics, pangenome analysis, and functional annotation studies.

| Output File | Description |
|---|---|
| `*.gff` | Master annotation file containing all annotated genomic features |
| `*.gbk` | GenBank formatted annotation file |
| `*.faa` | Protein sequences of predicted coding regions |
| `*.ffn` | Nucleotide sequences of predicted genes |
| `*.fna` | Nucleotide FASTA file of contig sequences |
| `*.fsa` | FASTA formatted genome assembly file |
| `*.tbl` | Annotation feature table used for GenBank submissions |
| `*.tsv` | Tab-separated summary of annotated features |
| `*.txt` | Summary statistics of genome annotation |
| `*.err` | Log file containing warnings and annotation-related issues |
| `*.log` | Execution log containing runtime information |
| `*.sqn` | Sequin submission file for NCBI genome submissions |

## **Citation**

Seemann T.

Prokka: rapid prokaryotic genome annotation.

Bioinformatics. 2014;30(14):2068–2069.

https://doi.org/10.1093/bioinformatics/btu153